# The dual problem

Substitute the two stationarity equations back into the Lagrangian. This removes $\mathbf{w}$ and $b$ and leaves a problem in the multipliers alone.

Because $\mathbf{w} = \sum_j \alpha_j y_j \mathbf{x}_j$,

$$
\|\mathbf{w}\|^2
= \sum_i \sum_j \alpha_i \alpha_j y_i y_j \, \mathbf{x}_i\cdot\mathbf{x}_j
$$

The constraint term also contains $\mathbf{w}$. After the algebra below, those two copies of the double sum cancel halfway, and the bias term disappears because $\sum_i \alpha_i y_i = 0$.


## The substitution

$$
\begin{aligned}
\mathcal{L}
&= \frac{1}{2}\|\mathbf{w}\|^2
- \sum_i \alpha_i y_i \mathbf{w}\cdot\mathbf{x}_i
- b\sum_i \alpha_i y_i
+ \sum_i \alpha_i
\end{aligned}
$$

The bias coefficient is zero by stationarity. Also

$$
\sum_i \alpha_i y_i \mathbf{w}\cdot\mathbf{x}_i
= \mathbf{w}\cdot\mathbf{w}
= \|\mathbf{w}\|^2
$$

so

$$
\mathcal{L}
= \frac{1}{2}\|\mathbf{w}\|^2 - \|\mathbf{w}\|^2 + \sum_i \alpha_i
= \sum_i \alpha_i - \frac{1}{2}\sum_i\sum_j \alpha_i\alpha_j y_i y_j \, \mathbf{x}_i\cdot\mathbf{x}_j
$$

The dual problem maximizes this over multipliers that stay nonnegative and balanced:

$$
\begin{aligned}
\underset{\boldsymbol{\alpha}}{\operatorname{maximize}} \quad
& \sum_i \alpha_i
- \frac{1}{2}\sum_i\sum_j \alpha_i \alpha_j y_i y_j \, \mathbf{x}_i\cdot\mathbf{x}_j \\
\text{subject to} \quad
& \alpha_i \ge 0,
\qquad \sum_i \alpha_i y_i = 0
\end{aligned}
$$

Only dot products of training points appear. That fact is what lets a kernel replace the dot product in a later lesson.


## Numbers for the three points

Order the points $(2,0)$, $(0,1)$, $(0,-1)$. Their dot products are

$$
\begin{aligned}
(2,0)\cdot(2,0) &= 4 \\
(2,0)\cdot(0,1) &= 0 \\
(2,0)\cdot(0,-1) &= 0 \\
(0,1)\cdot(0,1) &= 1 \\
(0,1)\cdot(0,-1) &= -1 \\
(0,-1)\cdot(0,-1) &= 1
\end{aligned}
$$

With $\alpha = (1/2,\ 1/4,\ 1/4)$ and $y = (+1,-1,-1)$,

$$
\sum_i \alpha_i = \frac{1}{2} + \frac{1}{4} + \frac{1}{4} = 1
$$

The double sum equals $\|\mathbf{w}\|^2 = 1$, so half of it is $1/2$. The dual objective is

$$
1 - \frac{1}{2} = \frac{1}{2}
$$

The primal objective was also $1/2$. The two problems agree. For a feasible hard-margin problem the constraints are linear and a separating line with room to spare exists, so this agreement is strong duality, not a coincidence of one example.


In [1]:
# Build the dual objective from dot products and compare it with the primal.
from fractions import Fraction

points = [(Fraction(2), Fraction(0)), (Fraction(0), Fraction(1)), (Fraction(0), Fraction(-1))]
labels = [Fraction(1), Fraction(-1), Fraction(-1)]
alpha = [Fraction(1, 2), Fraction(1, 4), Fraction(1, 4)]

quadratic = Fraction(0)
for i in range(3):
    for j in range(3):
        dot = points[i][0] * points[j][0] + points[i][1] * points[j][1]
        quadratic += alpha[i] * alpha[j] * labels[i] * labels[j] * dot
dual_objective = sum(alpha) - Fraction(1, 2) * quadratic
primal_objective = Fraction(1, 2)  # (1/2) * ||(1, 0)||^2
assert quadratic == 1
assert dual_objective == Fraction(1, 2)
assert dual_objective == primal_objective
print("sum of alphas", sum(alpha))
print("quadratic form", quadratic)
print("dual objective", dual_objective)


sum of alphas 1
quadratic form 1
dual objective 1/2


## A multiplier that is too large fails the balance

Suppose every multiplier is $1/2$. Then $\sum \alpha_i y_i = 1/2 - 1/2 - 1/2 = -1/2 \ne 0$. This vector is not dual-feasible, even though each entry is positive. The balance constraint is easy to forget because it is only one equation.


In [2]:
# Equal multipliers look plausible and still break the bias constraint.
from fractions import Fraction

labels = [1, -1, -1]
bad = [Fraction(1, 2), Fraction(1, 2), Fraction(1, 2)]
balance = sum(a * y for a, y in zip(bad, labels))
assert balance == Fraction(-1, 2)
good = [Fraction(1, 2), Fraction(1, 4), Fraction(1, 4)]
assert sum(a * y for a, y in zip(good, labels)) == 0
print("unbalanced sum", balance)


unbalanced sum -1/2


## A pitfall

The dual maximizes. The primal minimizes. They are not the same expression with different letters. At the optimum their values are equal, so you can check a proposed answer by computing both. Away from the optimum they differ.

## What to carry forward

Training a hard-margin SVM is a quadratic program in the multipliers. On the three-point example the optimal dual value is $1/2$, matching the primal.
